# Topic: ML: Missing Value Imputation

## Definition (30-second explanation)
* Missing value imputation is the process of replacing absent data (NaNs) with estimated or statistical values.
* Because most machine learning algorithms cannot handle missing values natively, imputation prevents errors and avoids the extreme data loss of simply dropping rows.

## Why Interviewers Ask This
* **To test for Data Leakage:** It is the #1 place junior candidates leak test data into the training set (by calculating global means before splitting).
* **To evaluate statistical intuition:** Do you blindly use the mean, or do you check for skewness to use the median?
* **To check business reasoning:** Do you understand that *why* data is missing (e.g., high earners skipping the salary field) is often a predictive signal itself?

## Core Concepts
* **MCAR (Missing Completely At Random):** Missingness is totally random (e.g., a sensor glitches 1% of the time).
* **MAR (Missing At Random):** Missingness depends on *other* observed features (e.g., younger users are less likely to fill out phone numbers).
* **MNAR (Missing Not At Random):** Missingness depends on the missing value itself (e.g., very sick patients are too sick to report their health status).
* **Missing Indicator (`was_missing`):** A binary column created before imputation to explicitly tell the model that a value was originally absent.

## When to Use
* **Mean Imputation:** For strictly normally distributed, continuous numerical data without outliers.
* **Median Imputation:** For skewed numerical data (e.g., income, prices, age).
* **Mode/Constant Imputation:** For categorical data (e.g., replacing with the most frequent class, or a new class like 'Unknown').
* **KNN/Iterative Imputation:** When features have complex relationships and simple univariate statistics destroy the data's underlying patterns.

## Advantages
* Prevents the massive loss of data that occurs when dropping rows with any missing values (`dropna()`).
* Advanced techniques (Iterative Imputer) can highly accurately reconstruct data by modeling it against all other features.
* Adding a missing indicator preserves the signal of MNAR data.

## Limitations
* Mean/Median imputation artificially reduces the variance of the feature and ignores feature interactions.
* KNN Imputation is computationally expensive ($O(N^2)$) and scales poorly to massive datasets.
* Bad imputation introduces bias, teaching the model false patterns.

## Common Comparisons
* **Mean vs. Median:** Mean is sensitive to outliers (skew); median is robust to outliers.
* **SimpleImputer vs. KNNImputer:** Simple looks at one column at a time (univariate); KNN looks at row similarity across multiple columns (multivariate).

## Common Interview Traps
* **The Data Leakage Trap:** Fitting your imputer on the *entire* dataset before doing `train_test_split`. You must fit on train, and transform train/test.
* **Ignoring the "Why":** Blindly imputing without asking if the missingness has business meaning.
* **Imputing the Target Variable:** You should almost never impute the target (`y`) variable in the training set; usually, those rows must be dropped.

## Python / SQL Syntax (if applicable)

    # Scikit-learn Best Practice
    from sklearn.impute import SimpleImputer
    
    # 1. Initialize
    imputer = SimpleImputer(strategy='median')
    
    # 2. FIT on Train, TRANSFORM on Train
    X_train['income'] = imputer.fit_transform(X_train[['income']])
    
    # 3. TRANSFORM ONLY on Test (Using Train's Median)
    X_test['income'] = imputer.transform(X_test[['income']])

## 45-Second Interview Answer
"Missing value imputation is necessary because most ML algorithms cannot process NaNs. My standard approach is to first understand *why* the data is missing—is it MCAR, MAR, or MNAR? For skewed numerical data, I use median imputation, and for complex relationships, I might use a multivariate approach like KNN. The two most critical rules I follow are: always fit the imputer strictly on the training set to prevent data leakage, and strongly consider adding a 'was_missing' binary indicator so the model can learn if the missingness itself holds predictive power."

## Example Questions:

### Q1. What is the difference between MCAR, MAR, and MNAR missing data types?
* **Ideal Interview Answer:** MCAR (Missing Completely At Random) means the missingness has no relationship with any data, like a random sensor failure. MAR (Missing At Random) means the missingness can be explained by *other* observed variables, like men being less likely to disclose their medical history. MNAR (Missing Not At Random) means the missingness is tied to the value itself—for instance, people with massive debts refusing to fill out a 'total debt' field.
* **Common Mistakes:** Confusing MAR and MNAR. (MAR depends on *other* columns; MNAR depends on the *missing column itself*).
* **Likely Follow-up:** If you suspect data is MNAR, how does that change your imputation strategy? *(Answer: Simply imputing the median destroys the signal. You must add a binary 'was_missing' flag before imputing so the model knows the value was deliberately hidden).*

### Q2. When would you use median imputation instead of mean imputation?
* **Ideal Interview Answer:** I use median imputation when the numerical feature is highly skewed or contains extreme outliers. The mean is heavily influenced by outliers—for example, one billionaire in a dataset will pull the mean income up drastically, causing me to impute an unrealistically high value for an average missing user. The median is robust to these outliers.
* **Common Mistakes:** Stating you use median for categorical variables (you use mode/most_frequent for that).
* **Likely Follow-up:** How would you quickly verify if a feature is skewed in Python? *(Answer: By looking at the difference between the mean and median using `df.describe()`, or visually plotting a histogram).*

### Q3. What is the risk of dropping all rows with missing values?
* **Ideal Interview Answer:** Dropping rows (`dropna`) can drastically reduce the size of your training data, leading to underfitting. More importantly, if the data is not MCAR (Completely At Random), dropping rows introduces severe selection bias. For example, if you drop all rows with missing income, and mostly low-income users skipped that field, your model will be heavily biased toward high-income users.
* **Common Mistakes:** Assuming dropping rows is always bad. (It's actually acceptable if the target variable is missing, or if the missingness is strictly MCAR and affects < 5% of a massive dataset).
* **Likely Follow-up:** If a feature has 90% missing values, should you impute it or drop the column entirely? *(Answer: Usually drop the column entirely, unless the 10% present is a highly predictive sparse signal).*

### Q4. How do you impute missing values correctly in a cross-validation pipeline?
* **Ideal Interview Answer:** You must ensure the imputer is fit *inside* the cross-validation loop on the training folds only, and then applied to the validation fold. If you impute before calling `cross_val_score`, the validation folds leak into the training means/medians. In Python, I use a `sklearn.pipeline.Pipeline` combining the `SimpleImputer` and the estimator to guarantee this strict isolation automatically.
* **Common Mistakes:** Saying "I use `fit_transform` on train and `transform` on test." (While true for standard splits, CV requires the use of a Pipeline to handle the rotating folds correctly).
* **Likely Follow-up:** Can you write the syntax for creating that sklearn Pipeline?

### Q5. What is KNN imputation and what is its main disadvantage?
* **Ideal Interview Answer:** KNN imputation replaces a missing value by finding the 'k' most similar rows (nearest neighbors) based on other features, and averaging their values. It is much more accurate than simple mean imputation because it captures multivariate relationships. However, its main disadvantage is computational complexity ($O(N^2)$); it requires calculating distances across the entire dataset, making it very slow and unscalable for massive datasets.
* **Common Mistakes:** Forgetting that KNN requires feature scaling (standardization/normalization) beforehand, because distance metrics (like Euclidean) are highly sensitive to different scales.
* **Likely Follow-up:** Because KNN is so slow, what is a faster algorithm-based alternative for multivariate imputation? *(Answer: Iterative Imputer / MICE, which models each feature as a function of the others using algorithms like Ridge or Bayesian Ridge regression).*

## Practice Questions:

### Q1:
**You are given a Pandas DataFrame df containing user data, and it has already been split into X_train and X_test. The income column has NaNs.**

- Using Python (either Pandas or Scikit-learn), write the code to accomplish the following on X_train and X_test:
1. Create a binary indicator column called income_was_missing (1 if missing, 0 if not).
2. Impute the missing values in the income column using the median.
3. Ensure absolute strictness against data leakage.

In [17]:
# Data:
import pandas as pd
import numpy as np

# Mock Data setup
X_train = pd.DataFrame({'age': [25, 30, 45, 35], 'income': [50000, np.nan, 120000, 60000]})
X_test = pd.DataFrame({'age': [28, 40], 'income': [np.nan, 80000]})

In [18]:
# Approach 1: Pandas (Explicit and clear)
X_train['income_was_missing'] = np.where(X_train['income'].isna(), 1, 0)
X_test['income_was_missing']  = np.where(X_test['income'].isna(), 1, 0)
  
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')
  
X_train['income'] = imputer.fit_transform(X_train[['income']])
X_test['income']  = imputer.transform(X_test[['income']])

In [19]:
X_train

,age,income,income_was_missing
0,25,50000.0,0
1,30,60000.0,1
2,45,120000.0,0
3,35,60000.0,0


In [20]:
X_test

,age,income,income_was_missing
0,28,60000.0,1
1,40,80000.0,0


In [21]:
# Data:
import pandas as pd
import numpy as np

# Mock Data setup
X_train = pd.DataFrame({'age': [25, 30, 45, 35], 'income': [50000, np.nan, 120000, 60000]})
X_test = pd.DataFrame({'age': [28, 40], 'income': [np.nan, 80000]})

# Approach 2: Sklearn Native (Best for Pipelines)
imputer = SimpleImputer(strategy='median', add_indicator=True)
X_train_imputed = imputer.fit_transform(X_train[['income']])
X_test_imputed= imputer.transform(X_test[['income']])

In [23]:
X_train_imputed

array([[5.0e+04, 0.0e+00],
       [6.0e+04, 1.0e+00],
       [1.2e+05, 0.0e+00],
       [6.0e+04, 0.0e+00]])

In [24]:
X_test_imputed

array([[6.e+04, 1.e+00],
       [8.e+04, 0.e+00]])

**Explanation:** "I create the indicator column first before the NaNs are replaced. Then, to strictly prevent data leakage, I initialize the imputer and call fit_transform on the training set to learn the median, but I strictly call transform on the test set so it utilizes the training median."

**Common Mistakes:** Imputing the data before creating the indicator (resulting in an indicator column of all zeros). Calling fit_transform on X_test, which leaks test distribution statistics into the evaluation.

**Likely Follow-up:** If we wanted to put this into an sklearn.pipeline.Pipeline, which of the two approaches (Pandas vs add_indicator=True) would be better? (Answer: The Sklearn native add_indicator=True approach, as Pipelines require all steps to have fit and transform methods).